In [ ]:
import os
import shutil

# 定义源目录和目标文件夹
source_directory = "/Users/yanminli/Documents/Study/causal inference/08 Datasets/ACIC-2018/scaling"  # 替换为你的文件所在目录
factuals_folder = "factuals"
counterfactuals_folder = "counterfactuals"

# 创建目标文件夹（如果不存在）
if not os.path.exists(factuals_folder):
    os.makedirs(factuals_folder)
if not os.path.exists(counterfactuals_folder):
    os.makedirs(counterfactuals_folder)

# 遍历源目录中的所有文件
for filename in os.listdir(source_directory):
    # 获取文件的完整路径
    file_path = os.path.join(source_directory, filename)

    # 检查文件是否以 "_cf.csv" 结尾
    if filename.endswith("_cf.csv"):
        # 移动到 counterfactuals 文件夹
        shutil.move(file_path, os.path.join(counterfactuals_folder, filename))
    elif filename.endswith(".csv"):
        # 移动到 factuals 文件夹
        shutil.move(file_path, os.path.join(factuals_folder, filename))

print("文件分类完成！")

In [1]:
import argparse
import pandas as pd
import torch

import os
# import pyro
import numpy as np
print(os.getcwd())

import pandas as pd

from sklearn import preprocessing



def load_data(path_data, num_sheet):
    cf_path = path_data + 'counterfactuals/' + str(num_sheet) + '_cf' + '.csv'
    f_path = path_data + 'factuals/'+ str(num_sheet) + '.csv'
    print(cf_path)
    print(f_path)
    # append the CSV files
    df_cf = pd.read_csv(cf_path)
    print(df_cf)
    df_f = pd.read_csv(f_path)
    print(df_f)
    x_path = path_data + 'x.csv'
    df_x = pd.read_csv(x_path)
    
    
    x_raw = pd.read_csv(x_path, header=0, sep=',')
    tymu_table_cf = df_cf.to_numpy()[:, 1:]
    tymu_table_f = df_f.to_numpy()[:, 1:]
    print(tymu_table_cf.shape) 
    print(tymu_table_cf)
    print(tymu_table_f.shape) 
    
#     print(df_cf['sample_id'].values)
    
    sample_id_list = df_cf['sample_id'].values

    # Filter the DataFrame to get rows where sample_id matches the values in your list
    filtered_df = x_raw[x_raw['sample_id'].isin(sample_id_list)]
#     print(filtered_df)
    x_table = filtered_df.to_numpy()[:, 1:]
    print(x_table.shape) 
    print(x_table)
    
    
    cov_scalar = preprocessing.StandardScaler()
    y_out_scaler = preprocessing.StandardScaler()

    x = cov_scalar.fit_transform(x_table)
    print(np.min(x), np.max(x))
    y_out_scaler.fit(np.concatenate([df_cf['y0'].values, df_cf['y1'].values]).reshape(-1, 1))
    y_0 = y_out_scaler.transform(df_cf['y0'].values.reshape(-1, 1))
    y_1 = y_out_scaler.transform(df_cf['y1'].values.reshape(-1, 1))


    mu_out_scaler = preprocessing.StandardScaler()

    mu_out_scaler.fit(np.concatenate([df_cf['y0'].values, df_cf['y1'].values]).reshape(-1, 1))
    mu_0 = mu_out_scaler.transform(df_cf['y0'].values.reshape(-1, 1))
    mu_1 = mu_out_scaler.transform(df_cf['y1'].values.reshape(-1, 1))
    
    t = df_f['z'].values.reshape(-1, 1)
    full_data = np.concatenate((t,y_0,y_1,mu_0,mu_1, x) , axis=1)
    print(full_data.shape) #(4802, 87)
    
    
#     print('Start normalizing data.')
    norm_data_df = pd.DataFrame(full_data)


    # save
    norm_data_folder = './acic2018_norm_data/'
    if not os.path.exists(norm_data_folder):
        os.makedirs(norm_data_folder)

    norm_data_path = norm_data_folder + num_sheet + '.csv'
    norm_data_df.to_csv(norm_data_path, index=False)
    print(norm_data_path)
    
    full_table = full_data

    mask = np.ones(full_table.shape)
   
    mask[:, 3] = 0 
    mask[:, 4] = 0 


    for i in range(full_table.shape[0]):
        t = full_table[i, 0]
        if t == 0:
            mask[i, 2] = 0 # mask y1

        if t == 1:
            mask[i, 1] = 0 # mask y0

#     print('finish generating mask.')
    #print(mask)

    mask_df = pd.DataFrame(mask)

    # save
    mask_folder = './acic2018_mask/'
    if not os.path.exists(mask_folder):
        os.makedirs(mask_folder)

    mask_path = mask_folder + num_sheet + '.csv'
    mask_df.to_csv(mask_path, index=False)
    print(mask_path)
    
#     print('Finish generating mask.')
    print('============== Finish this dataset')

    


/Users/yanminli/PycharmProjects/pythonProject/DiffS4-main/data_acic2018


In [15]:
import os

def get_num_sheet():
    # 定义文件夹路径
    folder_path = "./factuals"  # 替换为你的文件夹路径

    # 获取文件夹中所有文件的名称
    file_names = os.listdir(folder_path)

    # 创建一个空列表来存储文件名（不含后缀）
    file_names_without_extension = []

    # 遍历文件名列表
    for file_name in file_names:
        # 使用 os.path.splitext() 分离文件名和扩展名
        name_without_extension, _ = os.path.splitext(file_name)
        file_names_without_extension.append(name_without_extension)
    return file_names_without_extension

In [2]:
# load_data(path_data = "./", num_sheet = '00ea30e866f141d9880d5824a361a76a')
load_data(path_data = "./", num_sheet = '4c051f192c0b4c7185753d610a0b3f73')
# sheet_names = get_num_sheet()
#
# for num_sheet in sheet_names:
#     load_data(path_data = "./", num_sheet = num_sheet)

./counterfactuals/4c051f192c0b4c7185753d610a0b3f73_cf.csv
./factuals/4c051f192c0b4c7185753d610a0b3f73.csv
      sample_id          y0          y1
0       1919492  300.521783  303.783329
1       1504599   23.526999   26.788544
2        679671 -679.620985 -676.359439
3       3245191   10.899693   14.161238
4       3141674 -189.183156 -185.921611
...         ...         ...         ...
9995    3212729    8.455890   11.717436
9996    3712968   17.341634   20.603180
9997    1483216   20.228053   23.489598
9998    1636528   13.180797   16.442343
9999     262284 -152.558218 -149.296672

[10000 rows x 3 columns]
      sample_id  z           y
0       1919492  0  300.521783
1       1504599  1   26.788544
2        679671  0 -679.620985
3       3245191  0   10.899693
4       3141674  1 -185.921611
...         ... ..         ...
9995    3212729  1   11.717436
9996    3712968  1   20.603180
9997    1483216  1   23.489598
9998    1636528  1   16.442343
9999     262284  0 -152.558218

[10000 rows x 3

In [17]:
# print(sheet_names)

['194fe0e3c1644d41a5085b92d2fe7e54', '1f8b1ffc247b45f884e95d8ca4989fd9', 'd19d22301f53457d93dfb4739daef014', 'ff695a5ff5464ee1b7deda74ba5425d7', '8bf594caa1664f32a160c028479edd81', 'e023372879a343909d803690228090d6', '5227bb5fc1324e31a2f3e12bce9135bf', 'a3c4c668a3f84ba3ac42bf78c3ca35b1', '715f33728d0545bba499db7a1050cb02', 'b2744e80e70446c2a9154113d78b88c7', 'e7739d6f6cdc4493a095b619a9ae415f', 'd926e5a55750403dbb92a04b62306065', 'd4ad7285da1248759990d167c7d1af0d', '2974def923b14e7884596f4d9ae506c7', '2fd4b7c574be477aadc095786d8dec24', '601568e58ba5487297fa9314ecf407d6', '851a503c9dfd48b588edd24f80dfc8b7', '53690f224aaa415bb2cd2e5a8656c099', '4c051f192c0b4c7185753d610a0b3f73', '6d2e2a79f5ec486a8702ab3c443cb88e', '9d5249efba244308a93f54d7ff7cad7b', 'e2c3a1727fab41719646ce5c9335a612', 'dca9ecaff8194a4b9ccd940893adb543', '2a9b9bcedbfb40b4aa2540e58e2488dc', '5abf34fc614d4d5f9a42024094637ed9', '30fdfdf9847b4b03ae6dfa0009655fb6', 'e2461564b0764aeaaf52e94fb9c67ad2', '00ea30e866f141d9880d5824a3

In [18]:
# ['a6c1b082d4984a4d9b3a604797c707f6', 'a386e1395acd439281351a5cfac0ddf3', '4e470ab95bd341978bfdee6c093659aa', '0a2adba672c7478faa7a47137a87a3ab', 'c93beaf85be34e5a89ec8ec7b1fcc146', '4dce9e2f89c143fc952ea33a98c0d4d2', '3461122a161c45138bcd39be934c52b5', 'f2e5cac9902246fba6e5a5c3b11d1605', '95baaf5faf2e421eb112e22c7fedfd7a', '5cc4cad434a74f20aa259898eb07af5d', 'd09f96200455407db569ae33fe06b0d3', '53166055ae024c2ca3959859dffdfaa2', '630bc1cb56204013be94f6a7d2766892', '2e474ffd411b4fb085245e72de71c19a', '6c046d00a1c744469b401a148e4c3540', 'f4c24afdc049400aac2aa409431321b2', 'ea8ec4f5364049a19cb6cf92df0e2593', '32be2325ea7148128c87bfc54eebd64c', '00994e15d85b4a1ea866e89dd0b543e3', '7e4dbafff9bc4714bc4950f086bae4a0', '536d93c5b7474f4d8f135ec2db59f851', '9d8c8568b0ac4ea4a123d89d0b057105', '945048ed2a24412785038ad4c4ea2446', 'c55e20ac7b8042c086e363321a75aa12']
import os

# 定义需要保留的文件名列表（不含.csv后缀）
allowed_files = [
    "00ea30e866f141d9880d5824a361a76a",
    "194fe0e3c1644d41a5085b92d2fe7e54",
    "1f8b1ffc247b45f884e95d8ca4989fd9",
    "2fd4b7c574be477aadc095786d8dec24",
    "4c051f192c0b4c7185753d610a0b3f73",
    "5227bb5fc1324e31a2f3e12bce9135bf",
    "53690f224aaa415bb2cd2e5a8656c099",
    "5abf34fc614d4d5f9a42024094637ed9",
    "601568e58ba5487297fa9314ecf407d6",
    "6d2e2a79f5ec486a8702ab3c443cb88e",
    "715f33728d0545bba499db7a1050cb02",
    "7a49ac2f2e0f4109b1adcc33b56bf0a9",
    "851a503c9dfd48b588edd24f80dfc8b7",
    "8bf594caa1664f32a160c028479edd81",
    "9d5249efba244308a93f54d7ff7cad7b",
    "a3c4c668a3f84ba3ac42bf78c3ca35b1",
    "b2744e80e70446c2a9154113d78b88c7",
    "d4ad7285da1248759990d167c7d1af0d",
    "d926e5a55750403dbb92a04b62306065",
    "dca9ecaff8194a4b9ccd940893adb543",
    "e023372879a343909d803690228090d6",
    "e2461564b0764aeaaf52e94fb9c67ad2",
    "e2c3a1727fab41719646ce5c9335a612",
    "ff695a5ff5464ee1b7deda74ba5425d7"
]

# 定义文件夹路径
folder1 = "./acic2018_norm_data"
folder2 = "./acic2018_mask"

# 遍历文件夹中的文件并删除不在列表中的文件
def clean_folder(folder_path):
    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            # 去掉.csv后缀
            file_id = filename[:-4]
            if file_id not in allowed_files:
                file_path = os.path.join(folder_path, filename)
                os.remove(file_path)
                print(f"Deleted: {file_path}")

# 处理两个文件夹
clean_folder(folder1)
clean_folder(folder2)

Deleted: ./acic2018_norm_data/d19d22301f53457d93dfb4739daef014.csv
Deleted: ./acic2018_norm_data/e7739d6f6cdc4493a095b619a9ae415f.csv
Deleted: ./acic2018_norm_data/2974def923b14e7884596f4d9ae506c7.csv
Deleted: ./acic2018_norm_data/2a9b9bcedbfb40b4aa2540e58e2488dc.csv
Deleted: ./acic2018_norm_data/30fdfdf9847b4b03ae6dfa0009655fb6.csv
Deleted: ./acic2018_norm_data/59096d7a6c4e42339afd4502f5f2f06c.csv
Deleted: ./acic2018_mask/d19d22301f53457d93dfb4739daef014.csv
Deleted: ./acic2018_mask/e7739d6f6cdc4493a095b619a9ae415f.csv
Deleted: ./acic2018_mask/2974def923b14e7884596f4d9ae506c7.csv
Deleted: ./acic2018_mask/2a9b9bcedbfb40b4aa2540e58e2488dc.csv
Deleted: ./acic2018_mask/30fdfdf9847b4b03ae6dfa0009655fb6.csv
Deleted: ./acic2018_mask/59096d7a6c4e42339afd4502f5f2f06c.csv


In [10]:
import csv

def check_none_values(file_path):
    with open(file_path, 'r') as file:
        reader = csv.reader(file)
        for i, row in enumerate(reader, start=1):
            if any(field is None or field == '' for field in row):
                print(f"Row {i} contains None or empty values")

# 调用函数
check_none_values('/Users/yanminli/PycharmProjects/pythonProject/DiffS4-main/data_acic2018/acic2018_norm_data/5227bb5fc1324e31a2f3e12bce9135bf.csv')

In [11]:
import pandas as pd

def check_nan_in_csv(file_path):
    # 读取 CSV 文件
    df = pd.read_csv(file_path)

    # 检查是否有 NaN 值
    if df.isnull().values.any():
        print("CSV file contains NaN values.")

        # 找出包含 NaN 值的行号
        nan_rows = df[df.isnull().any(axis=1)]
        print("Rows containing NaN values:")
        print(nan_rows)
    else:
        print("CSV file does not contain NaN values.")

# 调用函数
check_nan_in_csv('/Users/yanminli/PycharmProjects/pythonProject/DiffS4-main/data_acic2018/acic2018_norm_data/5227bb5fc1324e31a2f3e12bce9135bf.csv')

CSV file does not contain NaN values.


In [3]:
import pandas as pd


def drop_rows_with_nan(file_path1,file_path2, output_file_path1, output_file_path2):
    # 读取 CSV 文件
    df1 = pd.read_csv(file_path1)
    df2 = pd.read_csv(file_path2)

    # 找出包含 NaN 值的行号
    nan_rows = df1[df1.isnull().any(axis=1)].index.tolist()

    # 打印包含 NaN 值的行号
    if nan_rows:
        print("Rows containing NaN values (will be dropped):", nan_rows)
    else:
        print("No rows contain NaN values.")

    # 删除包含 NaN 值的行
    df1.drop(nan_rows, inplace=True)
    df2.drop(nan_rows, inplace=True)

    # 保存处理后的数据到新的 CSV 文件
    df1.to_csv(output_file_path1, index=False)
    df2.to_csv(output_file_path2, index=False)

# 调用函数
drop_rows_with_nan('/home/liam/pythonProject/DiffS4-main/data/acic2018/acic2018_norm_data/194fe0e3c1644d41a5085b92d2fe7e54.csv','/home/liam/pythonProject/DiffS4-main/data/acic2018/acic2018_mask/194fe0e3c1644d41a5085b92d2fe7e54.csv',
                   '/home/liam/pythonProject/DiffS4-main/data/acic2018/acic2018_norm_data/194fe0e3c1644d41a5085b92d2fe7e54.csv',
                   '/home/liam/pythonProject/DiffS4-main/data/acic2018/acic2018_mask/194fe0e3c1644d41a5085b92d2fe7e54.csv')

Rows containing NaN values (will be dropped): [4123]
